# Guia TNF: asistente turistico de Tenerife

Este notebook sera el entregable principal de la practica. En esta fase construiremos la base RAG del asistente: carga de la guia, chunking, embeddings, indice FAISS, recuperacion de contexto y evaluacion simple del retrieval con salida legible mediante `print`.


## 0. Preparacion del entorno

El notebook espera una variable `OPENAI_API_KEY`. Puedes cargarla desde un archivo `.env` situado junto al notebook. Los modelos y parametros principales se configuran mediante variables de entorno y constantes para facilitar pruebas controladas durante la fase de RAG.


In [4]:
# Si estas en un entorno limpio, descomenta esta celda.
# Despues de instalar, puede ser necesario reiniciar el kernel.
%pip install -q -U openai python-dotenv tiktoken langchain langchain-openai langchain-community langchain-text-splitters pypdf faiss-cpu ipykernel


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 0.1. Imports y configuracion comun

Esta celda concentra imports, rutas, modelos y parametros del pipeline RAG. Dejamos visibles tanto el modelo generativo como el de embeddings, junto con los parametros de generacion que usaremos en esta fase.


In [5]:
import getpass
import os
import statistics
import time
from pathlib import Path
from typing import Any
from typing_extensions import TypedDict

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
import tiktoken


C:\Users\Trabajo\AppData\Local\Temp\ipykernel_33844\694594729.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [6]:
def find_project_dir() -> Path:
    current = Path.cwd()
    if (current / "guia_tnf.ipynb").exists():
        return current
    return current


PROJECT_DIR = find_project_dir()
load_dotenv(PROJECT_DIR / ".env")

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

GENERATION_MODEL = os.getenv("OPENAI_GENERATION_MODEL", "gpt-4.1-mini")
FAST_MODEL = os.getenv(
    "OPENAI_FAST_MODEL",
    os.getenv("OPENAI_EVALUATION_MODEL", GENERATION_MODEL),
)
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

GENERATION_TEMPERATURE = 0.0
GENERATION_TOP_P = 1.0
GENERATION_MAX_OUTPUT_TOKENS = 800
RETRIEVAL_K_DEFAULT = 4

GUIDE_PATH = PROJECT_DIR / "TENERIFE.pdf"
VECTOR_STORE_ROOT = PROJECT_DIR / "vector_store" / "faiss_tenerife"
CHUNKING_STRATEGIES = {
    "small": {"chunk_size": 500, "chunk_overlap": 80},
    "medium": {"chunk_size": 900, "chunk_overlap": 150},
    "large": {"chunk_size": 1200, "chunk_overlap": 150},
}

llm = init_chat_model(
    GENERATION_MODEL,
    model_provider="openai",
    temperature=GENERATION_TEMPERATURE,
    top_p=GENERATION_TOP_P,
    max_tokens=GENERATION_MAX_OUTPUT_TOKENS,
)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

print("Directorio del proyecto:", PROJECT_DIR)
print("Guia turistica:", GUIDE_PATH)
print("Modelo generativo:", GENERATION_MODEL)
print("Modelo rapido:", FAST_MODEL)
print("Modelo de embeddings:", EMBEDDING_MODEL)
print("Temperature:", GENERATION_TEMPERATURE)
print("Top-p:", GENERATION_TOP_P)
print("Max output tokens:", GENERATION_MAX_OUTPUT_TOKENS)
print("Retrieval k por defecto:", RETRIEVAL_K_DEFAULT)
print("Directorio de indices:", VECTOR_STORE_ROOT)

if not GUIDE_PATH.exists():
    raise FileNotFoundError(f"No se encontro la guia turistica en {GUIDE_PATH}")


Directorio del proyecto: c:\Repositorios\practicaLLMs
Guia turistica: c:\Repositorios\practicaLLMs\TENERIFE.pdf
Modelo generativo: gpt-4.1-mini
Modelo rapido: gpt-4.1-mini
Modelo de embeddings: text-embedding-3-small
Temperature: 0.0
Top-p: 1.0
Max output tokens: 800
Retrieval k por defecto: 4
Directorio de indices: c:\Repositorios\practicaLLMs\vector_store\faiss_tenerife


## 1. Carga del PDF

Primero cargamos el PDF y limpiamos un poco el texto para que el chunking no indexe saltos de linea innecesarios. Conservamos los metadatos originales de pagina y anadimos un nombre de fuente estable para poder citar despues.


In [7]:
def load_tenerife_pdf(pdf_path: Path) -> list[Document]:
    """Carga el PDF de Tenerife y normaliza el texto pagina por pagina."""
    loader = PyPDFLoader(str(pdf_path))
    raw_docs = loader.load()
    cleaned_docs: list[Document] = []

    for doc in raw_docs:
        cleaned_text = "\n".join(
            line.strip() for line in doc.page_content.splitlines() if line.strip()
        )
        metadata = dict(doc.metadata)
        metadata["source_name"] = pdf_path.name
        cleaned_docs.append(Document(page_content=cleaned_text, metadata=metadata))

    return cleaned_docs


raw_docs = load_tenerife_pdf(GUIDE_PATH)
print("Paginas cargadas:", len(raw_docs))
print("Metadata primera pagina:", raw_docs[0].metadata)
print("Primeros caracteres de la primera pagina:\n")
print(raw_docs[0].page_content[:1200])


Paginas cargadas: 25
Metadata primera pagina: {'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2025-07-13T20:00:01+00:00', 'title': 'Microsoft Word - TENERIFE.docx', 'moddate': '2025-07-13T20:00:01+00:00', 'source': 'c:\\Repositorios\\practicaLLMs\\TENERIFE.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'source_name': 'TENERIFE.pdf'}
Primeros caracteres de la primera pagina:

TENERIFE – LUGARES DE INTERÉS
SITIOS QUE VER
ZONA NORTE
• Santa Cruz de Tenerife:
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa
Cruz sería:
- Aparcar en el aparcamiento del Parque Marítimo (ubicación).
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación).
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación).
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo
dirección Plaza Weyler - ubicación –; ir hacia el Parque García Sanabria -
ubicación -; y bajar de nuevo hacia Plaza de 

## 2. Chunking

Comparamos tres estrategias de chunking sobre la misma guia. Todas usan `RecursiveCharacterTextSplitter`, pero cambian `chunk_size` y `chunk_overlap`. Para cada estrategia calculamos estadisticas simples y mostramos ejemplos reales de chunk con sus metadatos.


In [8]:
def build_chunked_documents(
    docs: list[Document],
    chunk_size: int,
    chunk_overlap: int,
    strategy_name: str,
) -> list[Document]:
    """Parte una coleccion de documentos y anade metadatos de auditoria."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
    )
    splits = splitter.split_documents(docs)

    for chunk_id, doc in enumerate(splits):
        doc.metadata["chunk_id"] = chunk_id
        doc.metadata["chunk_size"] = chunk_size
        doc.metadata["chunk_overlap"] = chunk_overlap
        doc.metadata["strategy_name"] = strategy_name
        doc.metadata.setdefault("source_name", GUIDE_PATH.name)

    return splits


def print_chunk_summary(strategy_name: str, documents: list[Document]) -> None:
    """Estadisticas basicas:"""
    lengths = [len(doc.page_content) for doc in documents]
    print("=" * 80)
    print(f"ESTRATEGIA: {strategy_name}")
    print("Numero de chunks:", len(documents))
    print("Longitud minima:", min(lengths))
    print("Longitud maxima:", max(lengths))
    print("Longitud media:", round(statistics.mean(lengths), 2))
    print("Ejemplo de chunk 1:")
    print(documents[0].page_content[:700])
    print("Metadata:", documents[0].metadata)
    print()
    if len(documents) > 1:
        print("Ejemplo de chunk 2:")
        print(documents[1].page_content[:700])
        print("Metadata:", documents[1].metadata)


CHUNKED_DOCUMENTS: dict[str, list[Document]] = {}
for strategy_name, config in CHUNKING_STRATEGIES.items():
    chunked_docs = build_chunked_documents(
        raw_docs,
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        strategy_name=strategy_name,
    )
    CHUNKED_DOCUMENTS[strategy_name] = chunked_docs
    print_chunk_summary(strategy_name, chunked_docs)


ESTRATEGIA: small
Numero de chunks: 48
Longitud minima: 22
Longitud maxima: 500
Longitud media: 353.02
Ejemplo de chunk 1:
TENERIFE – LUGARES DE INTERÉS
SITIOS QUE VER
ZONA NORTE
• Santa Cruz de Tenerife:
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa
Cruz sería:
- Aparcar en el aparcamiento del Parque Marítimo (ubicación).
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación).
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación).
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo
Metadata: {'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2025-07-13T20:00:01+00:00', 'title': 'Microsoft Word - TENERIFE.docx', 'moddate': '2025-07-13T20:00:01+00:00', 'source': 'c:\\Repositorios\\practicaLLMs\\TENERIFE.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'source_name': 'TENERIFE.pdf', 'start_index': 0, 'chunk_id': 0, 'chunk_size': 500, 'chunk_overlap': 80, 'st

## 3. Embeddings e indice FAISS

A partir de cada coleccion de chunks construimos un indice FAISS persistido en disco. Asi podemos reutilizar el trabajo de indexacion sin tener que recalcular embeddings en cada ejecucion del notebook.


In [9]:
def build_faiss_index(documents: list[Document], persist_dir: Path, embedding_model: str):
    """Construye un indice FAISS, lo guarda en disco y devuelve metadatos de construccion."""
    persist_dir.mkdir(parents=True, exist_ok=True)
    start_time = time.perf_counter()
    embedding_backend = OpenAIEmbeddings(model=embedding_model)
    vector_store = FAISS.from_documents(documents, embedding_backend)
    vector_store.save_local(str(persist_dir))
    build_time_s = time.perf_counter() - start_time
    return {
        "vector_store": vector_store,
        "build_time_s": build_time_s,
        "persist_dir": persist_dir,
        "num_documents": len(documents),
    }


def load_or_build_faiss_index(
    strategy_name: str,
    force_rebuild: bool = False,
):
    """Carga un indice FAISS existente o lo construye si no existe."""
    persist_dir = VECTOR_STORE_ROOT / strategy_name
    if persist_dir.exists() and not force_rebuild:
        start_time = time.perf_counter()
        vector_store = FAISS.load_local(
            str(persist_dir),
            OpenAIEmbeddings(model=EMBEDDING_MODEL),
            allow_dangerous_deserialization=True,
        )
        load_time_s = time.perf_counter() - start_time
        return {
            "vector_store": vector_store,
            "build_time_s": 0.0,
            "load_time_s": load_time_s,
            "persist_dir": persist_dir,
            "num_documents": len(CHUNKED_DOCUMENTS[strategy_name]),
            "loaded_from_disk": True,
        }

    built_index = build_faiss_index(
        CHUNKED_DOCUMENTS[strategy_name],
        persist_dir,
        EMBEDDING_MODEL,
    )
    built_index["load_time_s"] = 0.0
    built_index["loaded_from_disk"] = False
    return built_index


INDEX_REGISTRY: dict[str, dict[str, Any]] = {}
for strategy_name in CHUNKING_STRATEGIES:
    index_payload = load_or_build_faiss_index(strategy_name)
    INDEX_REGISTRY[strategy_name] = index_payload
    print("=" * 80)
    print(f"INDICE: {strategy_name}")
    print("Persistido en:", index_payload["persist_dir"])
    print("Numero de chunks indexados:", index_payload["num_documents"])
    print("Cargado desde disco:", index_payload["loaded_from_disk"])
    print("Tiempo de construccion (s):", round(index_payload["build_time_s"], 3))
    print("Tiempo de carga (s):", round(index_payload["load_time_s"], 3))


INDICE: small
Persistido en: c:\Repositorios\practicaLLMs\vector_store\faiss_tenerife\small
Numero de chunks indexados: 48
Cargado desde disco: True
Tiempo de construccion (s): 0.0
Tiempo de carga (s): 0.3
INDICE: medium
Persistido en: c:\Repositorios\practicaLLMs\vector_store\faiss_tenerife\medium
Numero de chunks indexados: 31
Cargado desde disco: True
Tiempo de construccion (s): 0.0
Tiempo de carga (s): 0.277
INDICE: large
Persistido en: c:\Repositorios\practicaLLMs\vector_store\faiss_tenerife\large
Numero de chunks indexados: 28
Cargado desde disco: True
Tiempo de construccion (s): 0.0
Tiempo de carga (s): 0.285


## 4. Retrieval y respuesta con fuentes

El retriever base usa `similarity_search`. La respuesta final debe redactarse sin citas inline, pero anadiendo una seccion `Fuentes` al final con pagina y chunk para auditoria.


In [10]:
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Eres un asistente guia turistico de Tenerife. "
            "Responde en espanol usando solo el contexto recuperado. "
            "Si el contexto no contiene informacion suficiente, dilo claramente. "
            "No inventes datos. No uses citas inline. "
            "La aplicacion anadira una seccion final de fuentes por separado.",
        ),
        (
            "user",
            "Pregunta:\n{question}\n\nContexto recuperado:\n{context}\n\nRespuesta:",
        ),
    ]
)


def retrieve_documents(question: str, strategy_name: str, k: int) -> list[Document]:
    """Recupera documentos relevantes desde el indice de una estrategia."""
    vector_store = INDEX_REGISTRY[strategy_name]["vector_store"]
    return vector_store.similarity_search(question, k=k)


def retrieve_documents_with_trace(
    question: str,
    strategy_name: str,
    k: int,
) -> tuple[list[Document], float]:
    """Recupera documentos y mide la latencia de retrieval."""
    start_time = time.perf_counter()
    docs = retrieve_documents(question, strategy_name, k)
    latency_ms = (time.perf_counter() - start_time) * 1000
    return docs, latency_ms


def format_docs_with_sources(documents: list[Document]) -> str:
    """Formatea el contexto recuperado con marcas internas de fuente."""
    blocks = []
    for i, doc in enumerate(documents, start=1):
        source_name = doc.metadata.get("source_name", "fuente_desconocida")
        page = doc.metadata.get("page", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        blocks.append(
            f"[Fuente {i}: {source_name}, pagina {page}, chunk {chunk_id}]\n{doc.page_content}"
        )
    return "\n\n".join(blocks)


def build_sources_list(documents: list[Document]) -> list[dict[str, Any]]:
    """Extrae una lista compacta de fuentes para la respuesta final."""
    sources = []
    for doc in documents:
        sources.append(
            {
                "source": doc.metadata.get("source_name", "fuente_desconocida"),
                "page": doc.metadata.get("page", "?"),
                "chunk_id": doc.metadata.get("chunk_id", "?"),
            }
        )
    return sources


def extract_token_usage(message: Any) -> Any:
    """Intenta recuperar informacion de uso de tokens desde la respuesta del modelo."""
    usage_metadata = getattr(message, "usage_metadata", None)
    if usage_metadata:
        return usage_metadata

    response_metadata = getattr(message, "response_metadata", None)
    if isinstance(response_metadata, dict) and response_metadata.get("token_usage"):
        return response_metadata["token_usage"]

    return None


def answer_with_rag(
    question: str,
    strategy_name: str,
    k: int = RETRIEVAL_K_DEFAULT,
) -> dict[str, Any]:
    """Recupera contexto, genera respuesta y devuelve trazabilidad simple."""
    retrieved_docs, retrieval_latency_ms = retrieve_documents_with_trace(
        question,
        strategy_name,
        k,
    )
    context = format_docs_with_sources(retrieved_docs)
    messages = rag_prompt.invoke({"question": question, "context": context})

    generation_start = time.perf_counter()
    response = llm.invoke(messages)
    generation_latency_ms = (time.perf_counter() - generation_start) * 1000

    answer_text = response.content if isinstance(response.content, str) else str(response.content)
    sources = build_sources_list(retrieved_docs)
    source_lines = [
        f"- {item['source']} | pagina {item['page']} | chunk {item['chunk_id']}"
        for item in sources
    ]
    final_answer = answer_text.strip() + "\n\nFuentes:\n" + "\n".join(source_lines)

    return {
        "question": question,
        "answer": final_answer,
        "sources": sources,
        "retrieved_docs": retrieved_docs,
        "strategy_name": strategy_name,
        "k": k,
        "retrieval_latency_ms": round(retrieval_latency_ms, 2),
        "generation_latency_ms": round(generation_latency_ms, 2),
        "token_usage": extract_token_usage(response),
    }


sample_question = "Cual es una visita imprescindible en Tenerife para un primer viaje?"
sample_result = answer_with_rag(sample_question, strategy_name="medium", k=RETRIEVAL_K_DEFAULT)
print("Pregunta:", sample_result["question"])
print("Estrategia:", sample_result["strategy_name"])
print("k:", sample_result["k"])
print("Latencia retrieval (ms):", sample_result["retrieval_latency_ms"])
print("Latencia generacion (ms):", sample_result["generation_latency_ms"])
print("Token usage:", sample_result["token_usage"])
print()
print(sample_result["answer"])


Pregunta: Cual es una visita imprescindible en Tenerife para un primer viaje?
Estrategia: medium
k: 4
Latencia retrieval (ms): 1363.28
Latencia generacion (ms): 2178.06
Token usage: {'input_tokens': 937, 'output_tokens': 174, 'total_tokens': 1111, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

Una visita imprescindible en Tenerife para un primer viaje es el Parque Nacional del Teide. Subir al Teide es considerado esencial para conocer la isla. Se recomienda subir por la carretera TF24 desde La Laguna, haciendo una parada en el Mirador de La Tarta para disfrutar de un espectacular mar de nubes si el clima lo permite. Al llegar al Parador de las Cañadas del Teide, se puede aparcar y visitar el mirador de La Ruleta, desde donde se obtiene una de las vistas más famosas de Tenerife. Además, es posible subir al pico del Teide utilizando los teleféricos y visitar el Centro de Visitantes de El Portillo, que es gratuito. Para los ama

## 5. Evaluacion basica del RAG

Evaluar RAG puede hacerse de forma sofisticada, pero aqui usaremos una evaluacion pequena y entendible, muy parecida al enfoque de los notebooks previos. Para cada caso queremos observar si el retrieval trae contexto relacionado, si el primer chunk ya parece util y si la respuesta final incluye fuentes.


In [11]:
EVAL_CASES = [
    {
        "name": "teide_basico",
        "input": "Que ver en el Parque Nacional del Teide?",
        "expected_keywords": {"teide", "parque nacional", "volcan"},
    },
    {
        "name": "anaga_paisaje",
        "input": "Merece la pena visitar Anaga y que tipo de paisaje ofrece?",
        "expected_keywords": {"anaga", "bosque", "sender"},
    },
]

EVAL_STRATEGY = "large"
EVAL_K = 5


def keyword_hit(documents: list[Document], keywords: set[str], top_n: int | None = None) -> bool:
    """Comprueba de forma heuristica si aparecen palabras clave en los textos recuperados."""
    selected_docs = documents if top_n is None else documents[:top_n]
    combined_text = " ".join(doc.page_content.lower() for doc in selected_docs)
    return any(keyword.lower() in combined_text for keyword in keywords)


def evaluate_retrieval_cases(
    cases: list[dict[str, Any]],
    strategy_name: str,
    k: int,
) -> list[dict[str, Any]]:
    """Evalua retrieval y respuesta final en un conjunto pequeno de casos."""
    rows = []
    for case in cases:
        try:
            run_result = answer_with_rag(case["input"], strategy_name=strategy_name, k=k)
            retrieved_docs = run_result["retrieved_docs"]
            observed_sources = build_sources_list(retrieved_docs)
            retrieval_ok = keyword_hit(retrieved_docs, case["expected_keywords"])
            top_1_ok = keyword_hit(retrieved_docs, case["expected_keywords"], top_n=1)
            answer_has_sources = "Fuentes:" in run_result["answer"]
            output_preview = run_result["answer"][:240]
            error = None
            retrieval_latency_ms = run_result["retrieval_latency_ms"]
            generation_latency_ms = run_result["generation_latency_ms"]
        except Exception as exc:
            observed_sources = []
            retrieval_ok = False
            top_1_ok = False
            answer_has_sources = False
            output_preview = ""
            error = f"{type(exc).__name__}: {exc}"
            retrieval_latency_ms = None
            generation_latency_ms = None

        rows.append(
            {
                "case": case["name"],
                "question": case["input"],
                "expected_keywords": sorted(case["expected_keywords"]),
                "observed_sources": observed_sources,
                "retrieval_ok": retrieval_ok,
                "top_1_ok": top_1_ok,
                "answer_has_sources": answer_has_sources,
                "retrieval_latency_ms": retrieval_latency_ms,
                "generation_latency_ms": generation_latency_ms,
                "error": error,
                "output_preview": output_preview,
            }
        )

    return rows


def print_retrieval_evaluation(results: list[dict[str, Any]], strategy_name: str, k: int) -> None:
    """Imprime una lectura compacta de la evaluacion de retrieval."""
    print("=" * 80)
    print(f"EVALUACION retrieval | estrategia={strategy_name} | k={k}")
    for row in results:
        print("-" * 80)
        print("Caso:", row["case"])
        print("Pregunta:", row["question"])
        print("Keywords esperadas:", row["expected_keywords"])
        print("Fuentes observadas:", row["observed_sources"])
        print("Retrieval OK:", row["retrieval_ok"])
        print("Top 1 OK:", row["top_1_ok"])
        print("Respuesta con fuentes:", row["answer_has_sources"])
        print("Latencia retrieval (ms):", row["retrieval_latency_ms"])
        print("Latencia generacion (ms):", row["generation_latency_ms"])
        print("Error:", row["error"])
        print("Output preview:")
        print(row["output_preview"])

    total = len(results)
    retrieval_ok_count = sum(1 for row in results if row["retrieval_ok"])
    top_1_ok_count = sum(1 for row in results if row["top_1_ok"])
    sources_ok_count = sum(1 for row in results if row["answer_has_sources"])
    retrieval_latencies = [row["retrieval_latency_ms"] for row in results if row["retrieval_latency_ms"] is not None]
    generation_latencies = [row["generation_latency_ms"] for row in results if row["generation_latency_ms"] is not None]

    print("=" * 80)
    print("RESUMEN")
    print("Casos evaluados:", total)
    print("Retrieval OK:", retrieval_ok_count, "/", total)
    print("Top 1 OK:", top_1_ok_count, "/", total)
    print("Respuestas con fuentes:", sources_ok_count, "/", total)
    if retrieval_latencies:
        print("Latencia media retrieval (ms):", round(statistics.mean(retrieval_latencies), 2))
    if generation_latencies:
        print("Latencia media generacion (ms):", round(statistics.mean(generation_latencies), 2))


In [12]:
# Si quieres repetir las pruebas de retrieval, puedes cambiar EVAL_STRATEGY y EVAL_K arriba y volver a ejecutar esta celda.

# evaluation_results = evaluate_retrieval_cases(EVAL_CASES, strategy_name=EVAL_STRATEGY, k=EVAL_K)
# print_retrieval_evaluation(evaluation_results, strategy_name=EVAL_STRATEGY, k=EVAL_K)


## 6. Dialogo multiturno con memoria

En esta fase anadimos continuidad conversacional sobre el RAG. El historial se mantiene de forma explicita en memoria del notebook, se recorta por tokens para controlar el contexto y LangGraph organiza el flujo de preparar pregunta, recuperar contexto, generar respuesta y actualizar memoria.


In [13]:
CONVERSATION_STRATEGY = "large"
CONVERSATION_K = 3
MAX_HISTORY_TOKENS = 1200
MAX_CONTEXT_TOKENS = 2500
MAX_TOTAL_PROMPT_TOKENS = 5000

CONVERSATION_HISTORY: dict[str, list[dict[str, str]]] = {}


def encoding_for_model_name(model_name: str) -> tiktoken.Encoding:
    """Devuelve un encoding compatible con el modelo configurado."""
    try:
        return tiktoken.encoding_for_model(model_name)
    except KeyError:
        return tiktoken.get_encoding("o200k_base")


def rough_token_estimate(text: str) -> int:
    """Estima tokens de texto con tiktoken para controlar el contexto."""
    encoding = encoding_for_model_name(GENERATION_MODEL)
    return len(encoding.encode(text or ""))


def history_to_text(history: list[dict[str, str]]) -> str:
    """Convierte historial reciente en texto legible para el prompt."""
    lines = []
    for turn in history:
        role = turn.get("role", "unknown")
        content = turn.get("content", "")
        lines.append(f"{role}: {content}")
    return "\n".join(lines)


def trim_history_by_tokens(
    history: list[dict[str, str]],
    max_tokens: int = MAX_HISTORY_TOKENS,
) -> list[dict[str, str]]:
    """Conserva los turnos mas recientes sin superar un presupuesto de tokens."""
    selected: list[dict[str, str]] = []
    total_tokens = 0

    for turn in reversed(history):
        turn_text = f"{turn.get('role', '')}: {turn.get('content', '')}"
        turn_tokens = rough_token_estimate(turn_text)
        if selected and total_tokens + turn_tokens > max_tokens:
            break
        if not selected and turn_tokens > max_tokens:
            truncated = turn.get("content", "")[-3000:]
            selected.append({"role": turn.get("role", "unknown"), "content": truncated})
            break
        selected.append(turn)
        total_tokens += turn_tokens

    return list(reversed(selected))


def trim_context_by_tokens(
    context: list[Document],
    max_tokens: int = MAX_CONTEXT_TOKENS,
) -> list[Document]:
    """Conserva chunks recuperados mientras quepan en el presupuesto de contexto."""
    selected: list[Document] = []
    total_tokens = 0

    for doc in context:
        doc_tokens = rough_token_estimate(doc.page_content)
        if selected and total_tokens + doc_tokens > max_tokens:
            break
        selected.append(doc)
        total_tokens += doc_tokens
        if total_tokens >= max_tokens:
            break

    return selected


def print_conversation_history(conversation_id: str = "demo") -> None:
    """Imprime el historial guardado para una conversacion."""
    history = CONVERSATION_HISTORY.get(conversation_id, [])
    print("=" * 80)
    print("HISTORIAL:", conversation_id)
    for i, turn in enumerate(history, start=1):
        print("-" * 80)
        print(f"Turno {i} | {turn.get('role')}")
        print(turn.get("content"))


print("Estrategia conversacional:", CONVERSATION_STRATEGY)
print("k conversacional:", CONVERSATION_K)
print("Max history tokens:", MAX_HISTORY_TOKENS)
print("Max context tokens:", MAX_CONTEXT_TOKENS)
print("Max total prompt tokens:", MAX_TOTAL_PROMPT_TOKENS)


Estrategia conversacional: large
k conversacional: 3
Max history tokens: 1200
Max context tokens: 2500
Max total prompt tokens: 5000


In [14]:
conversation_rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Reformula la pregunta del usuario para que sea autonoma. "
            "Usa el historial solo si la pregunta depende de referencias previas."
            "Devuelve solo la pregunta reformulada, sin explicaciones.",
        ),
        (
            "user",
            "Historial reciente:\n{history}\n\nPregunta actual:\n{question}\n\nPregunta autonoma:",
        ),
    ]
)

conversation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Eres un asistente guia turistico de Tenerife. "
            "Responde en espanol usando solo el contexto recuperado de la guia. "
            "Si el contexto no contiene informacion suficiente, dilo claramente. "
            "Ten en cuenta el historial reciente para mantener continuidad. "
            "No inventes datos y no uses citas inline. "
            "La aplicacion anadira las fuentes al final.",
        ),
        (
            "user",
            "Historial reciente:\n{history}\n\nPregunta:\n{question}\n\n"
            "Pregunta autonoma usada para buscar:\n{standalone_question}\n\n"
            "Contexto recuperado:\n{context}\n\nRespuesta:",
        ),
    ]
)


class ConversationState(TypedDict):
    question: str
    standalone_question: str
    history: list[dict[str, str]]
    context: list[Document]
    answer: str
    sources: list[dict[str, Any]]
    retrieval_latency_ms: float
    generation_latency_ms: float
    prompt_tokens_estimate: int


def prepare_question(state: ConversationState) -> dict[str, Any]:
    """Reformula la pregunta usando historial reciente cuando sea necesario."""
    trimmed_history = trim_history_by_tokens(state.get("history", []), MAX_HISTORY_TOKENS)
    history_text = history_to_text(trimmed_history)

    if not trimmed_history:
        standalone_question = state["question"]
    else:
        messages = conversation_rewrite_prompt.invoke(
            {"history": history_text, "question": state["question"]}
        )
        response = llm.invoke(messages)
        standalone_question = response.content.strip()

    return {"standalone_question": standalone_question, "history": trimmed_history}


def retrieve_conversation_context(state: ConversationState) -> dict[str, Any]:
    """Recupera contexto para la pregunta autonoma."""
    start_time = time.perf_counter()
    retrieved_docs = retrieve_documents(
        state["standalone_question"],
        strategy_name=CONVERSATION_STRATEGY,
        k=CONVERSATION_K,
    )
    retrieval_latency_ms = (time.perf_counter() - start_time) * 1000
    trimmed_context = trim_context_by_tokens(retrieved_docs, MAX_CONTEXT_TOKENS)
    return {
        "context": trimmed_context,
        "sources": build_sources_list(trimmed_context),
        "retrieval_latency_ms": round(retrieval_latency_ms, 2),
    }


def generate_conversation_answer(state: ConversationState) -> dict[str, Any]:
    """Genera la respuesta completa para uso no interactivo del grafo."""
    history_text = history_to_text(state.get("history", []))
    context_text = format_docs_with_sources(state.get("context", []))
    prompt_text = (
        history_text
        + "\n"
        + state.get("question", "")
        + "\n"
        + state.get("standalone_question", "")
        + "\n"
        + context_text
    )
    prompt_tokens_estimate = rough_token_estimate(prompt_text)

    if prompt_tokens_estimate > MAX_TOTAL_PROMPT_TOKENS:
        trimmed_history = trim_history_by_tokens(state.get("history", []), MAX_HISTORY_TOKENS // 2)
        trimmed_context = trim_context_by_tokens(state.get("context", []), MAX_CONTEXT_TOKENS // 2)
        history_text = history_to_text(trimmed_history)
        context_text = format_docs_with_sources(trimmed_context)
        prompt_tokens_estimate = rough_token_estimate(
            history_text + "\n" + state.get("question", "") + "\n" + context_text
        )
    else:
        trimmed_context = state.get("context", [])

    messages = conversation_prompt.invoke(
        {
            "history": history_text,
            "question": state["question"],
            "standalone_question": state["standalone_question"],
            "context": context_text,
        }
    )
    start_time = time.perf_counter()
    response = llm.invoke(messages)
    generation_latency_ms = (time.perf_counter() - start_time) * 1000
    answer_text = response.content if isinstance(response.content, str) else str(response.content)

    return {
        "answer": answer_text.strip(),
        "context": trimmed_context,
        "sources": build_sources_list(trimmed_context),
        "generation_latency_ms": round(generation_latency_ms, 2),
        "prompt_tokens_estimate": prompt_tokens_estimate,
    }


def update_conversation_state(state: ConversationState) -> dict[str, Any]:
    """Devuelve el estado sin persistir; la persistencia se hace por conversation_id."""
    return state


conversation_graph_builder = StateGraph(ConversationState)
conversation_graph_builder.add_node("prepare_question", prepare_question)
conversation_graph_builder.add_node("retrieve", retrieve_conversation_context)
conversation_graph_builder.add_node("generate", generate_conversation_answer)
conversation_graph_builder.add_node("update_history", update_conversation_state)
conversation_graph_builder.add_edge(START, "prepare_question")
conversation_graph_builder.add_edge("prepare_question", "retrieve")
conversation_graph_builder.add_edge("retrieve", "generate")
conversation_graph_builder.add_edge("generate", "update_history")
conversation_graph = conversation_graph_builder.compile()

print("Grafo conversacional preparado.")


Grafo conversacional preparado.


In [15]:
def stream_answer_from_state(state: ConversationState) -> dict[str, Any]:
    """Genera la respuesta en streaming desde un estado ya preparado."""
    history_text = history_to_text(state.get("history", []))
    trimmed_context = trim_context_by_tokens(state.get("context", []), MAX_CONTEXT_TOKENS)
    context_text = format_docs_with_sources(trimmed_context)
    prompt_tokens_estimate = rough_token_estimate(
        history_text
        + "\n"
        + state.get("question", "")
        + "\n"
        + state.get("standalone_question", "")
        + "\n"
        + context_text
    )

    if prompt_tokens_estimate > MAX_TOTAL_PROMPT_TOKENS:
        trimmed_history = trim_history_by_tokens(state.get("history", []), MAX_HISTORY_TOKENS // 2)
        trimmed_context = trim_context_by_tokens(trimmed_context, MAX_CONTEXT_TOKENS // 2)
        history_text = history_to_text(trimmed_history)
        context_text = format_docs_with_sources(trimmed_context)
        prompt_tokens_estimate = rough_token_estimate(
            history_text + "\n" + state.get("question", "") + "\n" + context_text
        )

    messages = conversation_prompt.invoke(
        {
            "history": history_text,
            "question": state["question"],
            "standalone_question": state["standalone_question"],
            "context": context_text,
        }
    )

    print("Asistente: ", end="", flush=True)
    answer_parts: list[str] = []
    start_time = time.perf_counter()
    for chunk in llm.stream(messages):
        piece = chunk.content if isinstance(chunk.content, str) else str(chunk.content)
        if piece:
            answer_parts.append(piece)
            print(piece, end="", flush=True)
    generation_latency_ms = (time.perf_counter() - start_time) * 1000
    print()

    sources = build_sources_list(trimmed_context)
    print("\nFuentes:")
    for source in sources:
        print(f"- {source['source']} | pagina {source['page']} | chunk {source['chunk_id']}")

    return {
        "answer": "".join(answer_parts).strip(),
        "sources": sources,
        "generation_latency_ms": round(generation_latency_ms, 2),
        "prompt_tokens_estimate": prompt_tokens_estimate,
        "context": trimmed_context,
    }


def chat_with_memory_stream(question: str, conversation_id: str = "demo") -> dict[str, Any]:
    """Ejecuta un turno conversacional con memoria, RAG y streaming."""
    history = CONVERSATION_HISTORY.get(conversation_id, [])
    initial_state: ConversationState = {
        "question": question,
        "standalone_question": question,
        "history": trim_history_by_tokens(history, MAX_HISTORY_TOKENS),
        "context": [],
        "answer": "",
        "sources": [],
        "retrieval_latency_ms": 0.0,
        "generation_latency_ms": 0.0,
        "prompt_tokens_estimate": 0,
    }

    prepared_state = initial_state
    print("=" * 80)
    print("Usuario:", question)
    for step in conversation_graph.stream(initial_state, stream_mode="updates"):
        if "prepare_question" in step:
            prepared_state.update(step["prepare_question"])
            print("Pregunta autonoma:", prepared_state["standalone_question"])
        if "retrieve" in step:
            prepared_state.update(step["retrieve"])
            print("Chunks recuperados:", len(prepared_state["context"]))
            print("Latencia retrieval (ms):", prepared_state["retrieval_latency_ms"])
            break

    streamed = stream_answer_from_state(prepared_state)
    final_answer = streamed["answer"]
    updated_history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": final_answer},
    ]
    CONVERSATION_HISTORY[conversation_id] = trim_history_by_tokens(
        updated_history,
        MAX_HISTORY_TOKENS,
    )

    return {
        "answer": final_answer,
        "sources": streamed["sources"],
        "history": CONVERSATION_HISTORY[conversation_id],
        "retrieval_latency_ms": prepared_state["retrieval_latency_ms"],
        "generation_latency_ms": streamed["generation_latency_ms"],
        "prompt_tokens_estimate": streamed["prompt_tokens_estimate"],
        "standalone_question": prepared_state["standalone_question"],
    }


print("Funcion chat_with_memory_stream lista.")


Funcion chat_with_memory_stream lista.


In [16]:
# Demo multiturno. Ejecuta estas llamadas cuando quieras probar la fase 3.

CONVERSATION_HISTORY["demo"] = []
turn_1 = chat_with_memory_stream("Que me recomiendas ver en el Teide?", conversation_id="demo")
turn_2 = chat_with_memory_stream("Y cuanto tiempo deberia reservar para esa visita?", conversation_id="demo")
turn_3 = chat_with_memory_stream("Y por que dicen que Gran Canaria es mas bonita?", conversation_id="demo")
print_conversation_history("demo")


Usuario: Que me recomiendas ver en el Teide?
Pregunta autonoma: Que me recomiendas ver en el Teide?
Chunks recuperados: 3
Latencia retrieval (ms): 298.84
Asistente: Te recomiendo subir al Parque Nacional del Teide por la carretera TF24 desde La Laguna, donde puedes hacer una parada en el Mirador de La Tarta para disfrutar de un espectacular mar de nubes si el clima lo permite. Al llegar al Parador de las Cañadas del Teide, puedes aparcar y visitar el mirador de La Ruleta, que ofrece una de las vistas más emblemáticas de la isla. 

Si quieres subir hasta el pico del Teide, puedes usar los teleféricos del Teide. Además, para más información, puedes visitar el Centro de Visitantes de El Portillo, que es gratuito. Una experiencia muy especial es subir de noche cuando el cielo esté despejado, ya que desde el Teide se puede observar uno de los cielos estrellados más impresionantes del mundo.

Fuentes:
- TENERIFE.pdf | pagina 14 | chunk 15
- TENERIFE.pdf | pagina 8 | chunk 7
- TENERIFE.pdf | 